In [1]:
import pandas as pd
import json
import itertools
import numpy as np
import networkx as nx
import operator
import os

/Users/katerina/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Networks of co-occuring hashtags for sampling and network analysis

In [3]:
### FUNCTIONS

# 1. Word counts
def word_count(tag):
    return len(tag.split())  # splits on spaces and counts words

# 2. Remove intersectionality from the hashtags list
def remove_intersectionality(tags):
    return [tag for tag in tags if tag.strip().lower() != 'intersectionality']

# 3. Get hashtag co-occurences

def hashtag_cooccurences(df, hashtag, year = None, top_n= None):
    #filtered_df = df.copy()
    
    # filter by hashtag
    filtered_df = df[
        (df['source'].str.lower() == hashtag) | (df['target'].str.lower() == hashtag)
    ]
    
    # filter by year
    if year:
        filtered_df = filtered_df[filtered_df['year'] == year]
    
    
    return (
        filtered_df.
        sort_values(by='count', ascending=False).
        reset_index(drop=True).
        head(top_n)
    )

In [4]:
## FOR JSON

with open("intersectionality_matches.json") as f: # or merged_intersectionality_all for the sampling 
    data = json.load(f)

# Extract only 'tags' and 'date' in order to use less memory

df = pd.DataFrame([
    {'tags': entry['tags'], 'date': entry['date']} 
    for entry in data
])

### 1. Creating edgelists of hashtags

In [5]:
# Create a new column with year

# Ensure 'date' is in datetime format

df['date'] = pd.to_datetime(df['date'])

# Extract year from the date column

df['year'] = df['date'].dt.year

In [ ]:
# Remove intersectionality from the hashtags

df['tags_new'] = df['tags'].apply(remove_intersectionality)

In [8]:
# Creating edge-list

# Efficiently create the pairs, not using iterrows()

pairs = [
    (tag1.lower(), tag2.lower(), year)
    for hashtags, year in zip(df['tags_new'], df['year']) 
    for tag1, tag2 in itertools.combinations(hashtags, 2)
]

# Construct the DataFramefrom the pairs

hashtag_df = pd.DataFrame(pairs, columns=['source', 'target', 'year'])


# Ensure (A, B) and (B, A) are treated the same

hashtag_df[['source', 'target']] = np.sort(hashtag_df[['source', 'target']], axis=1)

# Group by the sorted (source, target) pairs and count occurrences
hashtag_df =  pd.DataFrame({'count': hashtag_df.groupby(['source', 'target', 'year']).size()}).reset_index()

# Rename the count column for clarity

hashtag_df.rename(columns={'size': 'count'}, inplace=True)


hashtag_df = hashtag_df[(hashtag_df['source'].apply(word_count) <= 3) & (hashtag_df['target'].apply(word_count) <= 3)]


In [ ]:
hashtag_df.to_csv("edgelist_no_intersectionality.csv", index=False, encoding='utf-8')

In [15]:
# Save as separate cvs in order to import it to gephi

# Create output folder

output_dir = "gephi_exports_full_dataset"

os.makedirs(output_dir, exist_ok=True)

df = clean_hashtag_df.copy()

df.rename(columns={
    'source': 'Source',
    'target': 'Target',
    'count': 'Weight'
}, inplace=True)

# Group by year and export one file per year

for year in df['year'].unique():
    yearly_df = df[df['year'] == year]
    filename = f"{output_dir}/edges_{int(year)}.csv"
    yearly_df.to_csv(filename, index=False)


### 2. Calculating eigenvector centrality and intersections of top 15% hashtags for two years

In [ ]:
# Create a dictionary to store eigenvector centrality by year

eigenvector_centralities = {}

for year in hashtag_df['year'].unique():
    
    year_df = hashtag_df[hashtag_df['year'] == year]
    
    g = nx.from_pandas_edgelist(year_df, source='source', target='target', edge_attr='count')
    
    eigenvector_centrality = nx.eigenvector_centrality(g)
    
    eigenvector_centralities[year] = eigenvector_centrality
    

# Create a dictionary to store the top 15% and sorted eigenvector centralities by year

top_15_percent_eigenvector_centralities = {}
        
for year, centrality in eigenvector_centralities.items():
    eigenvector_sorted = sorted(centrality.items(), key=operator.itemgetter(1), reverse=True)
    top_n = int(len(eigenvector_sorted) * 0.15)  # ensures at least 1 hashtag
    top_15_percent_eigenvector_centralities[year] = eigenvector_sorted[:top_n]


# Convert to sets to enable set operations

top_15_percent_eigenvector_sets = {}

for year, centrality in top_15_percent_eigenvector_centralities.items():
    top_15_percent_eigenvector_sets[year] = set([x[0] for x in centrality])

In [ ]:
# example

top_15_percent_eigenvector_centralities[2015]

In [ ]:
# Sets of 2-year windows

set_13to14 = top_15_percent_eigenvector_sets[2013] & top_15_percent_eigenvector_sets[2014] 

set_14to15 = top_15_percent_eigenvector_sets[2014] & top_15_percent_eigenvector_sets[2015] 

set_15to16 = top_15_percent_eigenvector_sets[2015] & top_15_percent_eigenvector_sets[2016] 

set_16to17 = top_15_percent_eigenvector_sets[2016] & top_15_percent_eigenvector_sets[2017] 

set_17to18 = top_15_percent_eigenvector_sets[2017] & top_15_percent_eigenvector_sets[2018] 

set_18to19 = top_15_percent_eigenvector_sets[2018] & top_15_percent_eigenvector_sets[2019]

set_19to20 = top_15_percent_eigenvector_sets[2019] & top_15_percent_eigenvector_sets[2020] 

set_20to21 = top_15_percent_eigenvector_sets[2020] & top_15_percent_eigenvector_sets[2021] 

set_21to22 = top_15_percent_eigenvector_sets[2021] & top_15_percent_eigenvector_sets[2022] 

set_22to23 = top_15_percent_eigenvector_sets[2022] & top_15_percent_eigenvector_sets[2023] 

In [ ]:
#this is the correct!!

# 2-year windows

moving_windows = {"2013-2014": set_13to14,
                  "2014-2015": set_14to15, 
                  "2015-2016": set_15to16, 
                  "2016-2017": set_16to17, 
                  "2017-2018": set_17to18, 
                  "2018-2019": set_18to19,
                  "2019-2020": set_19to20,
                  "2020-2021": set_20to21,
                  "2021-2022": set_21to22,
                  "2022-2023": set_22to23,
}

# Precompute sets from other windows

other_window_sets = {
    k: set().union(*(v for wk, v in moving_windows.items() if wk != k))
    for k in moving_windows
}

unique_tags_per_window = {}

non_unique_tags_per_window = {}

for window_label, current_set in moving_windows.items():
    
    unique_to_window = current_set - other_window_sets[window_label] # only present in the current window
    
    non_unique_new_tags = current_set - unique_to_window # new hashtags that are not unique to the current window
    
    unique_tags_per_window[window_label] = unique_to_window
    
    non_unique_tags_per_window[window_label] = non_unique_new_tags
